# AI工学101 — 第21回

## 決定木：条件分岐をデータから学習する

第20回では、**汎化・過学習・正則化**を学びました。

ここから少し面白くなるぞ💪
今まで主に、

[
y = Wx+b
]

を中心としたモデルを使ってきました。

今日はその世界から一歩出て、

> **「もし特徴量が○○より小さければA、そうでなければB」**

という**条件分岐そのものを学習するモデル**を扱います。

それが **Decision Tree（決定木）** です。

---

# 🎯 今日のゴール

今日は次のことをできるようにします。

* 決定木がどんなモデルなのか説明できる
* `DecisionTreeClassifier` を使える
* 木の深さ `max_depth` の意味を理解する
* 決定木でも過学習が起こることを実験する
* `plot_tree()` で学習した木を可視化する
* 「モデルの構造そのものを見る」という発想を身につける

---

# 📖 講義：約20分

## 1. 決定木とは？

例えば、

```text
花びらの長さ > 5cm？
       /        \
     Yes         No
     /             \
  Class A        Class B
```

のように、

**質問を順番に繰り返して分類するモデル**です。

人間が書くなら、

```python
if petal_length > 5:
    class = 1
else:
    class = 0
```

のようなもの。

ただし決定木では、

> **どの特徴量を、どの値で分割すれば分類がうまくいくか**

をデータから自動的に決めます。

---

# 🧠 2. 「学習するif文」

ここが今日の核心。

普通のプログラムでは、

```python
if x > 5:
    ...
```

という条件を**人間が書きます**。

決定木では、

```text
どの特徴量？
↓
どの閾値？
↓
どちらに分ける？
```

を**学習アルゴリズムが決めます**。

つまり、

> **決定木は「学習するif文の集合」と考えるとかなり直感的。**

---

# 🌳 3. Treeの構造

決定木はこんな構造になります。

```text
              ┌── 条件A？
              │
          ┌── Yes
          │
      Root Node
          │
          └── No
              │
              └── 条件B？
                    │
                 ┌──┴──┐
                 Yes   No
                  ↓     ↓
               Class1 Class2
```

一番上を

**Root（根）**

と呼びます。

途中の条件を

**Node（ノード）**

最後の分類結果を

**Leaf（葉）**

と呼びます。

---

# 💻 実習1：Irisデータ

今回も `iris` を使います。

```python
from sklearn.datasets import load_iris

iris = load_iris()

X = iris.data
y = iris.target
```

特徴量名も見てみましょう。

```python
print(
    iris.feature_names
)
```

概ね、

```text
sepal length
sepal width
petal length
petal width
```

です。

---

# 💻 実習2：train/test分割

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

---

# 💻 実習3：決定木を作る

```python
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    random_state=42
)
```

学習。

```python
model.fit(
    X_train,
    y_train
)
```

予測。

```python
pred = model.predict(
    X_test
)
```

評価。

```python
from sklearn.metrics import accuracy_score

print(
    accuracy_score(
        y_test,
        pred
    )
)
```

これだけで決定木による分類ができます。

---

# 💻 実習4：木を可視化する

ここが決定木の楽しいところ。

```python
import matplotlib.pyplot as plt

from sklearn.tree import plot_tree
```

描画。

```python
plt.figure(
    figsize=(15, 10)
)

plot_tree(
    model,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True
)

plt.show()
```

実際に、

```text
petal length <= ...
```

のような条件分岐が表示されます。

---

# 👀 ここで観察

この木を見ながら、

> 「このモデルは何を根拠に分類しているんだろう？」

と考えてみてください。

線形回帰やロジスティック回帰では、

```text
coef_
```

を見ることで特徴量の影響を確認しました。

決定木では、

**木そのものを見ることができます。**

これが決定木の面白いところです。

---

# 💻 実習5：木の深さを制限する

ここで第20回の「過学習」が登場します。

まず、

```python
model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)
```

学習。

```python
model.fit(
    X_train,
    y_train
)
```

評価。

```python
train_acc = model.score(
    X_train,
    y_train
)

test_acc = model.score(
    X_test,
    y_test
)

print(
    "train:",
    train_acc
)

print(
    "test:",
    test_acc
)
```

---

# 🧠 `max_depth` とは？

決定木の

**最大深さ**

です。

例えば、

```text
max_depth=1
```

なら、

```text
質問1回
```

だけ。

```text
max_depth=2
```

なら、

```text
質問
 ↓
質問
```

くらいまで。

値を大きくすると、

```text
より複雑な条件分岐
```

を作れるようになります。

---

# 💻 実習6：深さを変えて比較する

ここが今日のメイン実験。

```python
depths = [
    1,
    2,
    3,
    4,
    5,
    10,
    None
]
```

それぞれ試します。

```python
for depth in depths:

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    train_acc = model.score(
        X_train,
        y_train
    )

    test_acc = model.score(
        X_test,
        y_test
    )

    print(
        "depth =", depth,
        "train =", train_acc,
        "test =", test_acc
    )
```

---

# 🔥 第20回との接続

ここで見てほしいのは、

```text
木が浅い
↓
モデルが単純

木が深い
↓
モデルが複雑
```

という関係です。

つまり、

```text
max_depth
    ↓
モデルの複雑さ
    ↓
過学習との関係
```

が見えてきます。

---

# 💻 実習7：深い木を見る

例えば、

```python
model = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)
```

として学習します。

```python
model.fit(
    X_train,
    y_train
)
```

そして、

```python
print(
    model.score(X_train, y_train)
)

print(
    model.score(X_test, y_test)
)
```

を比較。

さらに、

```python
plt.figure(
    figsize=(20, 12)
)

plot_tree(
    model,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True
)

plt.show()
```

として木を見ます。

---

# 🧠 何が起きている？

木を深くしていくと、

```text
データをより細かく分割
```

できます。

その結果、

```text
trainデータには非常によく合う
```

ようになります。

しかし、

```text
testデータ
```

では必ずしも良くなるとは限りません。

これは第20回の

> **過学習**

そのものです。

---

# 💻 実習8：特徴量の重要度

決定木では、

```python
model.feature_importances_
```

を見ることができます。

```python
print(
    model.feature_importances_
)
```

例えば、

```text
[0.0, 0.02, 0.55, 0.43]
```

のような結果だったとします。

これは、

```text
sepal length → ほぼ使っていない

sepal width → 少し

petal length → 重要

petal width → 重要
```

というように、

**どの特徴量が木の分割にどれくらい使われたか**

を見るための指標です。

---

# ⚠️ ただし「重要度＝因果関係」ではない

これは大事。

例えば、

```text
feature_importances_
```

が高いからといって、

> 「この特徴量が原因でクラスが決まっている」

とは言えません。

あくまで、

> **このモデルの予測に、その特徴量がどれくらい利用されたか**

という話です。

AI開発では、

**予測に役立つこと**と**因果関係があること**を混同しないようにします。

---

# 💻 実習9：Pipelineとの比較

決定木は、

```text
StandardScaler
```

が必須ではありません。

なぜなら、

```text
「値の大小」
```

を使って分割するからです。

例えば、

```text
身長 160
身長 170
身長 180
```

を、

```text
170以下？
```

のように分けるだけ。

単位を標準化しても、

**順序関係が基本的に変わらない**

からです。

なので、

```python
DecisionTreeClassifier()
```

をそのまま使えます。

これは、

> **すべてのモデルに同じ前処理をすればいいわけではない**

という重要な実務知識です。

---

# ✍️ 演習

今日の `iris` データを使います。

## 問1

`DecisionTreeClassifier` を使って、

```text
train/test
↓

fit
↓

predict
↓

accuracy
```

まで実行してください。

---

## 問2

次の4種類を比較してください。

```python
max_depth=1
max_depth=2
max_depth=3
max_depth=None
```

それぞれについて、

```text
train accuracy
test accuracy
```

を記録します。

---

## 問3

最もtest accuracyが高かった木を確認してください。

```python
plot_tree()
```

で可視化します。

---

## 問4

その木では、

> **最初にどの特徴量を使って分割しているか？**

を確認してください。

---

## 問5

`feature_importances_` を表示し、

特徴量名と対応させてください。

---

# 👾 ボス戦：モデルの複雑さを実験する

次のコードを完成させます。

```python
depths = [1, 2, 3, 4, 5, 6, None]

results = []

for depth in depths:

    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    train_acc = ...

    test_acc = ...

    results.append(
        (
            depth,
            train_acc,
            test_acc
        )
    )

for result in results:
    print(result)
```

そして、

```text
depth
train accuracy
test accuracy
```

の関係を観察してください。

### 最終問題

**「木を深くすれば性能が上がる」とは限らない理由を、今日の実験結果を使って説明する。**

これが今日のボス戦です。

---

# 🌱 今日のまとめ

今回の重要ポイントを一本につなげると、

```text
決定木
 ↓
条件分岐を学習
 ↓
木が深くなる
 ↓
表現力が上がる
 ↓
train性能が上がりやすい
 ↓
しかし深すぎると過学習
```

となります。

そして第20回で学んだ、

```text
モデルが単純すぎる
↓
Underfitting

        ↓

適切な複雑さ
↓
Generalization

        ↓

複雑すぎる
↓
Overfitting
```

が、**実際のモデル構造として目で見えるようになった**わけです。

さらに、

```text
Linear / Logistic Regression
        ↓
係数 W, b

Decision Tree
        ↓
条件分岐
```

という、**まったく違うモデルの考え方**にも触れました。

---

# 🧭 現在地：scikit-learn編

ここまで来ると、単なるAPI暗記ではなく、

```text
データ
 ↓
前処理
 ↓
特徴量
 ↓
モデル
 ↓
学習
 ↓
評価
 ↓
Cross Validation
 ↓
ハイパーパラメータ探索
 ↓
汎化・過学習
 ↓
モデル比較
```

という**機械学習開発の実験ループ**ができ始めています。

次はこの決定木を一人で頑張らせるのではなく、**複数の木を組み合わせる**方向へ進みます。

# 🔜 第22回

## Random Forest：たくさんの決定木を組み合わせる

次回は **Random Forest**。

一つの決定木は、

> 「このデータではこう分ける！」

と判断します。

では、

> **大量の決定木にそれぞれ少し違うデータ・特徴量を与えて、みんなに判断させたら？**

これが**アンサンブル学習**の基本アイデアです。

`RandomForestClassifier` を実装しながら、

* Bagging
* 多数決
* Random Forest
* `n_estimators`
* `max_depth`
* feature importance
* 決定木1本との性能比較

まで進みます。

ここから、**「一つの賢いモデル」から「複数の弱いモデルを組み合わせて強くする」**という、機械学習のもう一つの大きな設計思想に入っていきます。